In [ ]:
import os

print(os.listdir("/kaggle/input/datasets/aadimator"))

In [ ]:
!ls /kaggle/input/datasets/aadimator/nuscenes-mini

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini"

# Find the first CAM_FRONT image
cam_front_dir = os.path.join(data_path, "samples", "CAM_FRONT")
first_image = sorted(os.listdir(cam_front_dir))[0]

image_path = os.path.join(cam_front_dir, first_image)

print("Image:", first_image)
print("Path:", image_path)

img = Image.open(image_path)

plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
import os

lidar_dir = "/kaggle/input/datasets/aadimator/nuscenes-mini/samples/LIDAR_TOP"

files = sorted(os.listdir(lidar_dir))

print("Number of LiDAR samples:", len(files))
print("First file:", files[0])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lidar_path = "/kaggle/input/datasets/aadimator/nuscenes-mini/samples/LIDAR_TOP/n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151603547590.pcd.bin"

points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)

print("Shape:", points.shape)
print("First 5 points:")
print(points[:5])

In [ ]:
plt.figure(figsize=(10, 10))

plt.scatter(
    points[:, 0],
    points[:, 1],
    s=0.3
)

plt.xlabel("X (meters)")
plt.ylabel("Y (meters)")
plt.title("nuScenes LiDAR — Bird's-Eye View")
plt.axis("equal")
plt.show()

In [ ]:
import json

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini"

with open(f"{data_path}/v1.0-mini/sample.json") as f:
    samples = json.load(f)

with open(f"{data_path}/v1.0-mini/sample_annotation.json") as f:
    annotations = json.load(f)

print("Samples:", len(samples))
print("Annotations:", len(annotations))

print("\nFirst annotation:")
print(annotations[0])

In [ ]:
!pip install nuscenes-devkit

In [ ]:
import json
import os

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini/v1.0-mini"

def load_json(name):
    with open(os.path.join(data_path, name)) as f:
        return json.load(f)

samples = load_json("sample.json")
sample_data = load_json("sample_data.json")
calibrated_sensor = load_json("calibrated_sensor.json")
ego_pose = load_json("ego_pose.json")

print("samples:", len(samples))
print("sample_data:", len(sample_data))
print("calibrated sensors:", len(calibrated_sensor))
print("ego poses:", len(ego_pose))

In [ ]:
# # Build lookup tables
# sample_data_by_token = {
#     x["token"]: x for x in sample_data
# }

# # Take the first sample
# sample = samples[0]

# print("Sample token:", sample["token"])

# for sensor_name, token in sample["data"].items():
#     if sensor_name in ["CAM_FRONT", "LIDAR_TOP"]:
#         sd = sample_data_by_token[token]

#         print(f"\n{sensor_name}")
#         print("  filename:", sd["filename"])
#         print("  timestamp:", sd["timestamp"])
#         print("  calibrated_sensor_token:", sd["calibrated_sensor_token"])
#         print("  ego_pose_token:", sd["ego_pose_token"])

In [ ]:
sample_token = samples[0]["token"]

sample_sensor_data = [
    sd for sd in sample_data
    if sd["sample_token"] == sample_token
]

print("Sensor records:", len(sample_sensor_data))

for sd in sample_sensor_data:
    print(sd["filename"])

In [ ]:
# Get CAM_FRONT and LIDAR_TOP records
cam_sd = next(
    sd for sd in sample_sensor_data
    if "CAM_FRONT/" in sd["filename"]
)

lidar_sd = next(
    sd for sd in sample_sensor_data
    if "LIDAR_TOP/" in sd["filename"]
)

print("CAM_FRONT:")
print(cam_sd)

print("\nLIDAR_TOP:")
print(lidar_sd)

In [ ]:
# Build lookup dictionaries
calib_by_token = {
    x["token"]: x for x in calibrated_sensor
}

ego_by_token = {
    x["token"]: x for x in ego_pose
}

# Get calibration
cam_calib = calib_by_token[cam_sd["calibrated_sensor_token"]]
lidar_calib = calib_by_token[lidar_sd["calibrated_sensor_token"]]

# Get ego poses
cam_pose = ego_by_token[cam_sd["ego_pose_token"]]
lidar_pose = ego_by_token[lidar_sd["ego_pose_token"]]

print("=== CAM_FRONT calibration ===")
print(cam_calib)

print("\n=== LIDAR_TOP calibration ===")
print(lidar_calib)

print("\n=== CAM_FRONT ego pose ===")
print(cam_pose)

print("\n=== LIDAR_TOP ego pose ===")
print(lidar_pose)

In [ ]:
import numpy as np

lidar_path = os.path.join(
    "/kaggle/input/datasets/aadimator/nuscenes-mini",
    lidar_sd["filename"]
)

points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)

print("LiDAR points:", points.shape)

In [ ]:
import numpy as np

def quat_to_rotmat(q):
    """
    nuScenes quaternion format: [w, x, y, z]
    Returns 3x3 rotation matrix.
    """
    w, x, y, z = q

    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)]
    ])


def transform_points(points, translation, rotation):
    """
    Transform points using:
        p_out = R @ p + t
    """
    R = quat_to_rotmat(rotation)

    return points @ R.T + np.asarray(translation)


# LiDAR XYZ
points_xyz = points[:, :3]

# --------------------------------------------------
# LiDAR → LiDAR ego
# --------------------------------------------------
points_ego_lidar = transform_points(
    points_xyz,
    lidar_calib["translation"],
    lidar_calib["rotation"]
)

# --------------------------------------------------
# LiDAR ego → global
# --------------------------------------------------
points_global = transform_points(
    points_ego_lidar,
    lidar_pose["translation"],
    lidar_pose["rotation"]
)

# --------------------------------------------------
# Global → camera ego
# --------------------------------------------------
R_cam_pose = quat_to_rotmat(cam_pose["rotation"])

points_cam_ego = (
    points_global - np.asarray(cam_pose["translation"])
) @ R_cam_pose

# --------------------------------------------------
# Camera ego → camera
# --------------------------------------------------
R_cam_calib = quat_to_rotmat(cam_calib["rotation"])

points_camera = (
    points_cam_ego - np.asarray(cam_calib["translation"])
) @ R_cam_calib

print("LiDAR points:", points_xyz.shape)
print("Camera-frame points:", points_camera.shape)
print(points_camera[:5])

In [ ]:
# Camera intrinsics
K = np.array(cam_calib["camera_intrinsic"])

# Keep points in front of the camera
front_mask = points_camera[:, 2] > 0

points_front = points_camera[front_mask]

# Perspective projection
projected = points_front @ K.T

u = projected[:, 0] / projected[:, 2]
v = projected[:, 1] / projected[:, 2]

# Keep points inside the 1600x900 image
image_width = cam_sd["width"]
image_height = cam_sd["height"]

valid = (
    (u >= 0) & (u < image_width) &
    (v >= 0) & (v < image_height)
)

u = u[valid]
v = v[valid]

print("LiDAR points in front of camera:", len(points_front))
print("LiDAR points projected into image:", len(u))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Load CAM_FRONT image
image_path = os.path.join(
    "/kaggle/input/datasets/aadimator/nuscenes-mini",
    cam_sd["filename"]
)

image = Image.open(image_path)

plt.figure(figsize=(16, 9))

plt.imshow(image)

# Draw projected LiDAR points
plt.scatter(
    u,
    v,
    s=2,
    alpha=0.6
)

plt.xlim(0, image.width)
plt.ylim(image.height, 0)

plt.title("nuScenes CAM_FRONT + Projected LiDAR")
plt.axis("off")

plt.show()

# Lidar points in motion

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import os

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini"

# All 404 LiDAR keyframes
lidar_dir = os.path.join(data_path, "samples", "LIDAR_TOP")
lidar_files = sorted(os.listdir(lidar_dir))

print("Frames:", len(lidar_files))

# --------------------------------------------------
# Create figure
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 10))

ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_aspect("equal")

scatter = ax.scatter([], [], s=0.3)

title = ax.set_title("nuScenes LiDAR")

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")


def update(frame_idx):

    lidar_file = lidar_files[frame_idx]

    path = os.path.join(lidar_dir, lidar_file)

    points = np.fromfile(
        path,
        dtype=np.float32
    ).reshape(-1, 5)

    xyz = points[:, :3]

    # BEV
    x = xyz[:, 0]
    y = xyz[:, 1]

    scatter.set_offsets(np.column_stack((x, y)))

    title.set_text(
        f"nuScenes LiDAR — Frame {frame_idx + 1}/{len(lidar_files)}"
    )

    return scatter, title


anim = FuncAnimation(
    fig,
    update,
    frames=len(lidar_files),
    interval=100,
    blit=True
)

# --------------------------------------------------
# Save video
# --------------------------------------------------

output_path = "/kaggle/working/nuscenes_lidar.mp4"

writer = FFMpegWriter(
    fps=10,
    bitrate=4000
)

anim.save(
    output_path,
    writer=writer
)

plt.close(fig)

print("Saved:", output_path)

In [ ]:
from IPython.display import Video

Video(
    "/kaggle/working/nuscenes_lidar.mp4",
    embed=True
)

In [ ]:
import json
import os

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini"

with open(f"{data_path}/v1.0-mini/map.json") as f:
    maps = json.load(f)

print("Number of maps:", len(maps))

for m in maps:
    print(m)

# Adding map to lidar points

In [ ]:
from PIL import Image
import os

# Map path
map_path = os.path.join(
    data_path,
    "maps",
    "53992ee3023e5494b90c316c183be829.png"
)

# Allow large nuScenes map
Image.MAX_IMAGE_PIXELS = None

# Load map
map_img = Image.open(map_path)

In [ ]:
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

map_img = Image.open(map_path)

print("Map size:", map_img.size)
print("Mode:", map_img.mode)

In [ ]:
import matplotlib.pyplot as plt

# Downsample for visualization only
preview = map_img.copy()
preview.thumbnail((1200, 1200))

print("Preview size:", preview.size)

plt.figure(figsize=(12, 12))
plt.imshow(preview, cmap="gray")
plt.axis("off")
plt.title("Singapore One-North — nuScenes Semantic Map")
plt.show()

In [ ]:
import os
import json
import numpy as np
from PIL import Image

# ============================================================
# 1. Dataset paths
# ============================================================

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini"
meta_path = os.path.join(data_path, "v1.0-mini")

print("Dataset:", data_path)
print("Metadata:", meta_path)


# ============================================================
# 2. Load required nuScenes metadata
# ============================================================

with open(os.path.join(meta_path, "sample.json")) as f:
    samples = json.load(f)

with open(os.path.join(meta_path, "sample_data.json")) as f:
    sample_data = json.load(f)

with open(os.path.join(meta_path, "ego_pose.json")) as f:
    ego_poses = json.load(f)

with open(os.path.join(meta_path, "calibrated_sensor.json")) as f:
    calibrated_sensors = json.load(f)


# ============================================================
# 3. Map
# ============================================================

map_path = os.path.join(
    data_path,
    "maps",
    "53992ee3023e5494b90c316c183be829.png"
)

Image.MAX_IMAGE_PIXELS = None

map_img = Image.open(map_path)

map_width, map_height = map_img.size

print("\nMap:", map_img.size)


# ============================================================
# 4. Build fast lookup dictionaries
# ============================================================

pose_lookup = {
    p["token"]: p
    for p in ego_poses
}

calib_lookup = {
    c["token"]: c
    for c in calibrated_sensors
}


# ============================================================
# 5. Get all LIDAR_TOP keyframes
# ============================================================

lidar_records = sorted(
    [
        sd for sd in sample_data
        if sd["filename"].startswith("samples/LIDAR_TOP")
        and sd["is_key_frame"]
    ],
    key=lambda x: x["timestamp"]
)

print("LIDAR keyframes:", len(lidar_records))


# ============================================================
# 6. Global ego trajectory
# ============================================================

trajectory_global = np.array([
    pose_lookup[sd["ego_pose_token"]]["translation"][:2]
    for sd in lidar_records
])

print("Trajectory:", trajectory_global.shape)


# ============================================================
# 7. Global → map pixel coordinates
# ============================================================

resolution = 0.1

trajectory_px = (
    trajectory_global[:, 0] / resolution
)

trajectory_py = (
    -trajectory_global[:, 1] / resolution
    + map_height
)

print(
    "Pixel X:",
    trajectory_px.min(),
    "→",
    trajectory_px.max()
)

print(
    "Pixel Y:",
    trajectory_py.min(),
    "→",
    trajectory_py.max()
)


# ============================================================
# 8. Crop map around complete trajectory
# ============================================================

pad = 1500  # 150 meters

x_min = max(
    0,
    int(trajectory_px.min() - pad)
)

x_max = min(
    map_width,
    int(trajectory_px.max() + pad)
)

y_min = max(
    0,
    int(trajectory_py.min() - pad)
)

y_max = min(
    map_height,
    int(trajectory_py.max() + pad)
)

map_crop = map_img.crop(
    (x_min, y_min, x_max, y_max)
)

print("\nMap crop:")
print("x:", x_min, "→", x_max)
print("y:", y_min, "→", y_max)
print("Crop size:", map_crop.size)

print("\nREADY")

In [ ]:
# Load ego poses
with open(f"{meta_path}/ego_pose.json") as f:
    ego_poses = json.load(f)

# Find first sample's LIDAR_TOP record
first_sample_token = samples[0]["token"]

lidar_sd = next(
    sd for sd in sample_data
    if sd["sample_token"] == first_sample_token
    and sd["filename"].startswith("samples/LIDAR_TOP")
)

# Get its ego pose
lidar_pose = next(
    p for p in ego_poses
    if p["token"] == lidar_sd["ego_pose_token"]
)

print("Map dimensions (W, H):", map_img.size)
print("First LiDAR global translation:", lidar_pose["translation"])
print("First LiDAR global rotation:", lidar_pose["rotation"])

In [ ]:
import os

maps_dir = os.path.join(data_path, "maps")

print("Map directory contents:")
for f in sorted(os.listdir(maps_dir)):
    print(f)

In [ ]:
# nuScenes semantic map resolution
map_resolution = 0.1  # meters per pixel

map_width, map_height = map_img.size

x_global, y_global, _ = lidar_pose["translation"]

# nuScenes map raster convention:
# global origin is at the center of the raster
pixel_x = int(map_width / 2 + x_global / map_resolution)
pixel_y = int(map_height / 2 - y_global / map_resolution)

print("Global position:")
print(f"  x = {x_global:.3f} m")
print(f"  y = {y_global:.3f} m")

print("\nEstimated map pixel:")
print(f"  u = {pixel_x}")
print(f"  v = {pixel_y}")

print("\nInside map:",
      0 <= pixel_x < map_width and
      0 <= pixel_y < map_height)

In [ ]:
import os
import glob

nuscenes_pkg = "/usr/local/lib/python3.12/dist-packages/nuscenes"

matches = glob.glob(
    os.path.join(nuscenes_pkg, "**", "*map_mask*.py"),
    recursive=True
)

print(matches)

In [ ]:
resolution = 0.1

map_width, map_height = map_img.size

x_global, y_global, _ = lidar_pose["translation"]

px = round(x_global / resolution)
py = round(-y_global / resolution + map_height)

print("Global:", x_global, y_global)
print("Pixel:", px, py)
print("Map size:", map_width, map_height)
print("Inside map:", 0 <= px < map_width and 0 <= py < map_height)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 12))

plt.imshow(map_img, cmap="gray")

plt.scatter(
    px,
    py,
    s=100,
    c="red",
    marker="o",
    label="Ego vehicle"
)

plt.xlim(px - 1000, px + 1000)
plt.ylim(py + 1000, py - 1000)

plt.legend()
plt.title("nuScenes Map — Ego Vehicle Position")
plt.show()

In [ ]:
# Load first LiDAR point cloud
lidar_path = os.path.join(
    data_path,
    lidar_sd["filename"]
)

points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
points_xyz = points[:, :3]

# LiDAR -> ego -> global
points_ego = transform_points(
    points_xyz,
    lidar_calib["translation"],
    lidar_calib["rotation"]
)

points_global = transform_points(
    points_ego,
    lidar_pose["translation"],
    lidar_pose["rotation"]
)

print("LiDAR points:", points_xyz.shape)
print("Global points:", points_global.shape)
print("Global X range:", points_global[:, 0].min(), points_global[:, 0].max())
print("Global Y range:", points_global[:, 1].min(), points_global[:, 1].max())

In [ ]:
# Convert global LiDAR coordinates to map pixels
resolution = 0.1

lidar_px = np.round(points_global[:, 0] / resolution).astype(np.int32)
lidar_py = np.round(
    -points_global[:, 1] / resolution + map_height
).astype(np.int32)

# Keep only points inside the map
valid = (
    (lidar_px >= 0) & (lidar_px < map_width) &
    (lidar_py >= 0) & (lidar_py < map_height)
)

lidar_px = lidar_px[valid]
lidar_py = lidar_py[valid]

print("LiDAR points on map:", len(lidar_px))

In [ ]:
plt.figure(figsize=(12, 12))

plt.imshow(map_img, cmap="gray")

# LiDAR points
plt.scatter(
    lidar_px,
    lidar_py,
    s=1,
    alpha=0.35,
    label="LiDAR"
)

# Ego vehicle
plt.scatter(
    px,
    py,
    s=120,
    c="red",
    marker="o",
    label="Ego"
)

# Zoom around vehicle
margin = 1000

plt.xlim(px - margin, px + margin)
plt.ylim(py + margin, py - margin)

plt.legend()
plt.title("nuScenes Global LiDAR + HD Map")
plt.show()

In [ ]:
# Test transformation for the first LiDAR keyframe

sd = lidar_records[0]

# Load LiDAR
path = os.path.join(data_path, sd["filename"])

pts = np.fromfile(
    path,
    dtype=np.float32
).reshape(-1, 5)

xyz = pts[:, :3]

# Calibration
calib = next(
    c for c in calibrated_sensors
    if c["token"] == sd["calibrated_sensor_token"]
)

# Ego pose
pose = next(
    p for p in ego_poses
    if p["token"] == sd["ego_pose_token"]
)

# LiDAR -> ego
ego_pts = transform_points(
    xyz,
    calib["translation"],
    calib["rotation"]
)

# Ego -> global
global_pts = transform_points(
    ego_pts,
    pose["translation"],
    pose["rotation"]
)

# Global -> map pixels
test_px = global_pts[:, 0] / resolution
test_py = (
    -global_pts[:, 1] / resolution
    + map_height
)

print("LiDAR points:", len(xyz))
print("Global points:", global_pts.shape)
print("Map pixel X range:", test_px.min(), test_px.max())
print("Map pixel Y range:", test_py.min(), test_py.max())

In [ ]:
# ============================================================
# Select LiDAR frames belonging ONLY to the first scene
# ============================================================

first_scene_token = "cc8c0bf57f984915a77078b10eb33198"

# sample_token -> scene_token
sample_scene_lookup = {
    s["token"]: s["scene_token"]
    for s in samples
}

scene_lidar_records = sorted(
    [
        sd for sd in sample_data
        if sd["filename"].startswith("samples/LIDAR_TOP")
        and sd["is_key_frame"]
        and sample_scene_lookup.get(sd["sample_token"])
            == first_scene_token
    ],
    key=lambda x: x["timestamp"]
)

print("LiDAR frames in first scene:",
      len(scene_lidar_records))

print(
    "First timestamp:",
    scene_lidar_records[0]["timestamp"]
)

print(
    "Last timestamp:",
    scene_lidar_records[-1]["timestamp"]
)

In [ ]:
import cv2
import numpy as np
import os

# ============================================================
# 1. Ego trajectory for THIS scene only
# ============================================================

scene_trajectory_global = np.array([
    pose_lookup[sd["ego_pose_token"]]["translation"][:2]
    for sd in scene_lidar_records
])

scene_trajectory_px = (
    scene_trajectory_global[:, 0] / resolution
)

scene_trajectory_py = (
    -scene_trajectory_global[:, 1] / resolution
    + map_height
)

print("Scene trajectory:", scene_trajectory_global.shape)


# ============================================================
# 2. Crop map around THIS scene
# ============================================================

PAD_METERS = 100
pad_px = int(PAD_METERS / resolution)

scene_x_min = max(
    0,
    int(scene_trajectory_px.min()) - pad_px
)

scene_x_max = min(
    map_width,
    int(scene_trajectory_px.max()) + pad_px
)

scene_y_min = max(
    0,
    int(scene_trajectory_py.min()) - pad_px
)

scene_y_max = min(
    map_height,
    int(scene_trajectory_py.max()) + pad_px
)

scene_map = map_img.crop(
    (
        scene_x_min,
        scene_y_min,
        scene_x_max,
        scene_y_max
    )
)

scene_map_np = np.array(scene_map)

print("Scene map:", scene_map.size)


# ============================================================
# 3. Resize map once
# ============================================================

VIDEO_WIDTH = 1200

scale = VIDEO_WIDTH / scene_map.width

VIDEO_HEIGHT = int(
    scene_map.height * scale
)

scene_map_resized = cv2.resize(
    scene_map_np,
    (VIDEO_WIDTH, VIDEO_HEIGHT),
    interpolation=cv2.INTER_AREA
)

base_frame = cv2.cvtColor(
    scene_map_resized,
    cv2.COLOR_GRAY2BGR
)


# ============================================================
# 4. Trajectory → video coordinates
# ============================================================

traj_x = (
    (scene_trajectory_px - scene_x_min)
    * scale
).astype(np.int32)

traj_y = (
    (scene_trajectory_py - scene_y_min)
    * scale
).astype(np.int32)


# ============================================================
# 5. Video writer
# ============================================================

output_video = (
    "/kaggle/working/"
    "nuscenes_scene1_lidar_map.mp4"
)

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    10,
    (VIDEO_WIDTH, VIDEO_HEIGHT)
)

print(
    "Video resolution:",
    VIDEO_WIDTH,
    "x",
    VIDEO_HEIGHT
)


# ============================================================
# 6. Render 39 frames
# ============================================================

for frame_idx, sd in enumerate(scene_lidar_records):

    frame = base_frame.copy()

    # --------------------------------------------------------
    # Load LiDAR
    # --------------------------------------------------------

    lidar_path = os.path.join(
        data_path,
        sd["filename"]
    )

    points = np.fromfile(
        lidar_path,
        dtype=np.float32
    ).reshape(-1, 5)

    xyz = points[:, :3]


    # --------------------------------------------------------
    # LiDAR → Ego
    # --------------------------------------------------------

    calib = calib_lookup[
        sd["calibrated_sensor_token"]
    ]

    ego_points = transform_points(
        xyz,
        calib["translation"],
        calib["rotation"]
    )


    # --------------------------------------------------------
    # Ego → Global
    # --------------------------------------------------------

    pose = pose_lookup[
        sd["ego_pose_token"]
    ]

    global_points = transform_points(
        ego_points,
        pose["translation"],
        pose["rotation"]
    )


    # --------------------------------------------------------
    # Global → map pixels
    # --------------------------------------------------------

    px = (
        global_points[:, 0] / resolution
    )

    py = (
        -global_points[:, 1] / resolution
        + map_height
    )


    # --------------------------------------------------------
    # Map crop coordinates
    # --------------------------------------------------------

    px = (
        (px - scene_x_min) * scale
    ).astype(np.int32)

    py = (
        (py - scene_y_min) * scale
    ).astype(np.int32)


    # --------------------------------------------------------
    # Keep valid points
    # --------------------------------------------------------

    valid = (
        (px >= 0) &
        (px < VIDEO_WIDTH) &
        (py >= 0) &
        (py < VIDEO_HEIGHT)
    )

    px = px[valid]
    py = py[valid]


    # --------------------------------------------------------
    # Draw LiDAR
    # --------------------------------------------------------

    frame[py, px] = (
        255, 100, 0
    )


    # --------------------------------------------------------
    # Draw trajectory
    # --------------------------------------------------------

    if frame_idx > 0:

        trajectory = np.column_stack(
            (
                traj_x[:frame_idx + 1],
                traj_y[:frame_idx + 1]
            )
        )

        cv2.polylines(
            frame,
            [trajectory],
            False,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )


    # --------------------------------------------------------
    # Ego vehicle
    # --------------------------------------------------------

    ex = traj_x[frame_idx]
    ey = traj_y[frame_idx]

    cv2.circle(
        frame,
        (ex, ey),
        7,
        (0, 0, 255),
        -1
    )


    # --------------------------------------------------------
    # Frame information
    # --------------------------------------------------------

    cv2.putText(
        frame,
        f"Singapore One-North | "
        f"Frame {frame_idx + 1}/39",
        (25, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    writer.write(frame)


    if (
        frame_idx == 0
        or (frame_idx + 1) % 10 == 0
    ):
        print(
            f"Rendered {frame_idx + 1}/39"
        )


writer.release()

print("\nDONE")
print(output_video)

In [ ]:
# ============================================================
# ALL 404 nuScenes LiDAR SAMPLES
# Scene-aware map visualization
# ============================================================

import os
import json
import cv2
import numpy as np
from PIL import Image

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

FPS = 10
VIDEO_W = 1200
VIDEO_H = 900
RESOLUTION = 0.1
PAD_METERS = 100

output_video = "/kaggle/working/nuscenes_all_scenes_lidar_map.mp4"


# ============================================================
# 1. Load scene / log / map metadata
# ============================================================

with open(os.path.join(meta_path, "scene.json")) as f:
    scenes = json.load(f)

with open(os.path.join(meta_path, "log.json")) as f:
    logs = json.load(f)

with open(os.path.join(meta_path, "map.json")) as f:
    maps = json.load(f)


# ------------------------------------------------------------
# Fast lookups
# ------------------------------------------------------------

scene_lookup = {
    s["token"]: s
    for s in scenes
}

log_lookup = {
    l["token"]: l
    for l in logs
}

sample_lookup = {
    s["token"]: s
    for s in samples
}


# ============================================================
# 2. Group LIDAR keyframes by scene
# ============================================================

scene_lidar = {
    s["token"]: []
    for s in scenes
}

for sd in sample_data:

    if not sd["is_key_frame"]:
        continue

    if not sd["filename"].startswith(
        "samples/LIDAR_TOP"
    ):
        continue

    sample = sample_lookup.get(
        sd["sample_token"]
    )

    if sample is None:
        continue

    scene_token = sample["scene_token"]

    scene_lidar[scene_token].append(sd)


# Sort each scene chronologically

for scene_token in scene_lidar:

    scene_lidar[scene_token].sort(
        key=lambda x: x["timestamp"]
    )


print("Scenes:", len(scenes))

print(
    "Total LiDAR keyframes:",
    sum(len(v) for v in scene_lidar.values())
)


# ============================================================
# 3. Map lookup
# ============================================================

log_to_map = {}

for m in maps:

    for log_token in m["log_tokens"]:
        log_to_map[log_token] = m["filename"]


# ============================================================
# 4. Prepare video writer
# ============================================================

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (VIDEO_W, VIDEO_H)
)

if not writer.isOpened():
    raise RuntimeError(
        "Could not open video writer"
    )


# ============================================================
# 5. Process scenes
# ============================================================

total_frames = 0

for scene_idx, scene in enumerate(scenes):

    scene_token = scene["token"]

    records = scene_lidar[
        scene_token
    ]

    if not records:
        print(
            f"Skipping scene {scene_idx + 1}: "
            "no LiDAR frames"
        )
        continue


    # --------------------------------------------------------
    # Scene / log information
    # --------------------------------------------------------

    log = log_lookup[
        scene["log_token"]
    ]

    location = log["location"]

    map_filename = log_to_map.get(
        scene["log_token"]
    )

    print(
        f"\nScene {scene_idx + 1}/{len(scenes)}"
    )

    print(
        "Location:",
        location
    )

    print(
        "Frames:",
        len(records)
    )

    print(
        "Map:",
        map_filename
    )


    if map_filename is None:
        print("No map found — skipping.")
        continue


    # --------------------------------------------------------
    # Load relevant map
    # --------------------------------------------------------

    map_path = os.path.join(
        data_path,
        map_filename
    )

    Image.MAX_IMAGE_PIXELS = None

    map_img = Image.open(
        map_path
    )


    map_width, map_height = map_img.size


    # --------------------------------------------------------
    # Scene ego trajectory
    # --------------------------------------------------------

    scene_trajectory_global = np.array([
        pose_lookup[
            sd["ego_pose_token"]
        ]["translation"][:2]
        for sd in records
    ])


    trajectory_px = (
        scene_trajectory_global[:, 0]
        / RESOLUTION
    )

    trajectory_py = (
        -scene_trajectory_global[:, 1]
        / RESOLUTION
        + map_height
    )


    # --------------------------------------------------------
    # Crop map around THIS scene
    # --------------------------------------------------------

    pad_px = int(
        PAD_METERS / RESOLUTION
    )

    x_min = max(
        0,
        int(trajectory_px.min())
        - pad_px
    )

    x_max = min(
        map_width,
        int(trajectory_px.max())
        + pad_px
    )

    y_min = max(
        0,
        int(trajectory_py.min())
        - pad_px
    )

    y_max = min(
        map_height,
        int(trajectory_py.max())
        + pad_px
    )


    map_crop = map_img.crop(
        (
            x_min,
            y_min,
            x_max,
            y_max
        )
    )


    # --------------------------------------------------------
    # Resize map to fit fixed video canvas
    # --------------------------------------------------------

    crop_w, crop_h = map_crop.size

    scale = min(
        VIDEO_W / crop_w,
        VIDEO_H / crop_h
    )

    resized_w = int(
        crop_w * scale
    )

    resized_h = int(
        crop_h * scale
    )

    map_np = np.array(
        map_crop
    )

    map_resized = cv2.resize(
        map_np,
        (resized_w, resized_h),
        interpolation=cv2.INTER_AREA
    )

    base_frame = np.zeros(
        (VIDEO_H, VIDEO_W, 3),
        dtype=np.uint8
    )

    map_bgr = cv2.cvtColor(
        map_resized,
        cv2.COLOR_GRAY2BGR
    )

    # Center map on canvas

    offset_x = (
        VIDEO_W - resized_w
    ) // 2

    offset_y = (
        VIDEO_H - resized_h
    ) // 2

    base_frame[
        offset_y:
        offset_y + resized_h,
        offset_x:
        offset_x + resized_w
    ] = map_bgr


    # --------------------------------------------------------
    # Convert trajectory to video coordinates
    # --------------------------------------------------------

    traj_x = (
        (trajectory_px - x_min)
        * scale
        + offset_x
    ).astype(np.int32)

    traj_y = (
        (trajectory_py - y_min)
        * scale
        + offset_y
    ).astype(np.int32)


    # --------------------------------------------------------
    # Render scene frames
    # --------------------------------------------------------

    for frame_idx, sd in enumerate(records):

        frame = base_frame.copy()


        # ----------------------------------------------------
        # Load LiDAR
        # ----------------------------------------------------

        lidar_path = os.path.join(
            data_path,
            sd["filename"]
        )

        points = np.fromfile(
            lidar_path,
            dtype=np.float32
        ).reshape(-1, 5)

        xyz = points[:, :3]


        # ----------------------------------------------------
        # LiDAR → Ego
        # ----------------------------------------------------

        calib = calib_lookup[
            sd["calibrated_sensor_token"]
        ]

        ego_points = transform_points(
            xyz,
            calib["translation"],
            calib["rotation"]
        )


        # ----------------------------------------------------
        # Ego → Global
        # ----------------------------------------------------

        pose = pose_lookup[
            sd["ego_pose_token"]
        ]

        global_points = transform_points(
            ego_points,
            pose["translation"],
            pose["rotation"]
        )


        # ----------------------------------------------------
        # Global → Map pixels
        # ----------------------------------------------------

        px = (
            global_points[:, 0]
            / RESOLUTION
        )

        py = (
            -global_points[:, 1]
            / RESOLUTION
            + map_height
        )


        # ----------------------------------------------------
        # Map crop → video coordinates
        # ----------------------------------------------------

        px = (
            (px - x_min)
            * scale
            + offset_x
        ).astype(np.int32)

        py = (
            (py - y_min)
            * scale
            + offset_y
        ).astype(np.int32)


        # ----------------------------------------------------
        # Valid points
        # ----------------------------------------------------

        valid = (
            (px >= 0) &
            (px < VIDEO_W) &
            (py >= 0) &
            (py < VIDEO_H)
        )

        px = px[valid]
        py = py[valid]


        # ----------------------------------------------------
        # Draw LiDAR
        # ----------------------------------------------------

        frame[py, px] = (
            255, 100, 0
        )


        # ----------------------------------------------------
        # Draw trajectory
        # ----------------------------------------------------

        if frame_idx > 0:

            trajectory = np.column_stack(
                (
                    traj_x[:frame_idx + 1],
                    traj_y[:frame_idx + 1]
                )
            )

            cv2.polylines(
                frame,
                [trajectory],
                False,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )


        # ----------------------------------------------------
        # Ego vehicle
        # ----------------------------------------------------

        ex = traj_x[frame_idx]
        ey = traj_y[frame_idx]

        cv2.circle(
            frame,
            (ex, ey),
            8,
            (0, 0, 255),
            -1
        )


        # ----------------------------------------------------
        # Scene information
        # ----------------------------------------------------

        cv2.rectangle(
            frame,
            (10, 10),
            (550, 95),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            frame,
            f"Scene {scene_idx + 1}/{len(scenes)} | "
            f"{location}",
            (25, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Frame {frame_idx + 1}/{len(records)}",
            (25, 72),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )


        # ----------------------------------------------------
        # Write
        # ----------------------------------------------------

        writer.write(frame)

        total_frames += 1


    # --------------------------------------------------------
    # Release map memory before next scene
    # --------------------------------------------------------

    map_img.close()

    print(
        f"Completed scene "
        f"{scene_idx + 1}/{len(scenes)}"
    )


# ============================================================
# 6. Finish
# ============================================================

writer.release()

print("\n======================================")
print("ALL SCENES COMPLETE")
print("======================================")
print("Total frames:", total_frames)
print("FPS:", FPS)
print(
    "Duration:",
    round(total_frames / FPS, 2),
    "seconds"
)
print("Output:")
print(output_video)

In [ ]:
# ============================================================
# ALL 404 nuScenes LiDAR SAMPLES
# Scene-aware map visualization + 3D bounding boxes
# ============================================================

import os
import json
import cv2
import numpy as np
from PIL import Image

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

FPS = 10
VIDEO_W = 1200
VIDEO_H = 900
RESOLUTION = 0.1
PAD_METERS = 30

output_video = "/kaggle/working/nuscenes_all_scenes_lidar_map_boxes.mp4"


# ============================================================
# 1. Load scene / log / map metadata
# ============================================================

with open(os.path.join(meta_path, "scene.json")) as f:
    scenes = json.load(f)

with open(os.path.join(meta_path, "log.json")) as f:
    logs = json.load(f)

with open(os.path.join(meta_path, "map.json")) as f:
    maps = json.load(f)


# ------------------------------------------------------------
# Fast lookups
# ------------------------------------------------------------

scene_lookup = {
    s["token"]: s
    for s in scenes
}

log_lookup = {
    l["token"]: l
    for l in logs
}

sample_lookup = {
    s["token"]: s
    for s in samples
}


# ============================================================
# 2. Group LIDAR keyframes by scene
# ============================================================

scene_lidar = {
    s["token"]: []
    for s in scenes
}

for sd in sample_data:

    if not sd["is_key_frame"]:
        continue

    if not sd["filename"].startswith(
        "samples/LIDAR_TOP"
    ):
        continue

    sample = sample_lookup.get(
        sd["sample_token"]
    )

    if sample is None:
        continue

    scene_token = sample["scene_token"]

    scene_lidar[scene_token].append(sd)


# Sort each scene chronologically

for scene_token in scene_lidar:

    scene_lidar[scene_token].sort(
        key=lambda x: x["timestamp"]
    )


print("Scenes:", len(scenes))

print(
    "Total LiDAR keyframes:",
    sum(len(v) for v in scene_lidar.values())
)


# ============================================================
# 3. Map lookup
# ============================================================

log_to_map = {}

for m in maps:

    for log_token in m["log_tokens"]:
        log_to_map[log_token] = m["filename"]


# ============================================================
# 4. Prepare video writer
# ============================================================

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (VIDEO_W, VIDEO_H)
)

if not writer.isOpened():
    raise RuntimeError(
        "Could not open video writer"
    )


# ============================================================
# 5. Process scenes
# ============================================================

total_frames = 0

for scene_idx, scene in enumerate(scenes):

    scene_token = scene["token"]

    records = scene_lidar[
        scene_token
    ]

    if not records:

        print(
            f"Skipping scene {scene_idx + 1}: "
            "no LiDAR frames"
        )

        continue


    # --------------------------------------------------------
    # Scene / log information
    # --------------------------------------------------------

    log = log_lookup[
        scene["log_token"]
    ]

    location = log["location"]

    map_filename = log_to_map.get(
        scene["log_token"]
    )

    print(
        f"\nScene {scene_idx + 1}/{len(scenes)}"
    )

    print(
        "Location:",
        location
    )

    print(
        "Frames:",
        len(records)
    )

    print(
        "Map:",
        map_filename
    )


    if map_filename is None:

        print("No map found — skipping.")

        continue


    # --------------------------------------------------------
    # Load relevant map
    # --------------------------------------------------------

    map_path = os.path.join(
        data_path,
        map_filename
    )

    Image.MAX_IMAGE_PIXELS = None

    map_img = Image.open(
        map_path
    )

    map_width, map_height = map_img.size


    # --------------------------------------------------------
    # Scene ego trajectory
    # --------------------------------------------------------

    scene_trajectory_global = np.array([
        pose_lookup[
            sd["ego_pose_token"]
        ]["translation"][:2]
        for sd in records
    ])


    trajectory_px = (
        scene_trajectory_global[:, 0]
        / RESOLUTION
    )

    trajectory_py = (
        -scene_trajectory_global[:, 1]
        / RESOLUTION
        + map_height
    )


    # --------------------------------------------------------
    # Crop map around THIS scene
    # --------------------------------------------------------

    pad_px = int(
        PAD_METERS / RESOLUTION
    )

    x_min = max(
        0,
        int(trajectory_px.min())
        - pad_px
    )

    x_max = min(
        map_width,
        int(trajectory_px.max())
        + pad_px
    )

    y_min = max(
        0,
        int(trajectory_py.min())
        - pad_px
    )

    y_max = min(
        map_height,
        int(trajectory_py.max())
        + pad_px
    )


    map_crop = map_img.crop(
        (
            x_min,
            y_min,
            x_max,
            y_max
        )
    )


    # --------------------------------------------------------
    # Resize map to fit fixed video canvas
    # --------------------------------------------------------

    crop_w, crop_h = map_crop.size

    scale = min(
        VIDEO_W / crop_w,
        VIDEO_H / crop_h
    )

    resized_w = int(
        crop_w * scale
    )

    resized_h = int(
        crop_h * scale
    )

    map_np = np.array(
        map_crop
    )

    map_resized = cv2.resize(
        map_np,
        (resized_w, resized_h),
        interpolation=cv2.INTER_AREA
    )

    base_frame = np.zeros(
        (VIDEO_H, VIDEO_W, 3),
        dtype=np.uint8
    )

    map_bgr = cv2.cvtColor(
        map_resized,
        cv2.COLOR_GRAY2BGR
    )


    # Center map on canvas

    offset_x = (
        VIDEO_W - resized_w
    ) // 2

    offset_y = (
        VIDEO_H - resized_h
    ) // 2

    base_frame[
        offset_y:
        offset_y + resized_h,
        offset_x:
        offset_x + resized_w
    ] = map_bgr


    # --------------------------------------------------------
    # Convert trajectory to video coordinates
    # --------------------------------------------------------

    traj_x = (
        (trajectory_px - x_min)
        * scale
        + offset_x
    ).astype(np.int32)

    traj_y = (
        (trajectory_py - y_min)
        * scale
        + offset_y
    ).astype(np.int32)


    # ========================================================
    # Render scene frames
    # ========================================================

    for frame_idx, sd in enumerate(records):

        frame = base_frame.copy()


        # ----------------------------------------------------
        # Load LiDAR
        # ----------------------------------------------------

        lidar_path = os.path.join(
            data_path,
            sd["filename"]
        )

        points = np.fromfile(
            lidar_path,
            dtype=np.float32
        ).reshape(-1, 5)

        xyz = points[:, :3]


        # ----------------------------------------------------
        # LiDAR → Ego
        # ----------------------------------------------------

        calib = calib_lookup[
            sd["calibrated_sensor_token"]
        ]

        ego_points = transform_points(
            xyz,
            calib["translation"],
            calib["rotation"]
        )


        # ----------------------------------------------------
        # Ego → Global
        # ----------------------------------------------------

        pose = pose_lookup[
            sd["ego_pose_token"]
        ]

        global_points = transform_points(
            ego_points,
            pose["translation"],
            pose["rotation"]
        )


        # ----------------------------------------------------
        # Global → Map pixels
        # ----------------------------------------------------

        px = (
            global_points[:, 0]
            / RESOLUTION
        )

        py = (
            -global_points[:, 1]
            / RESOLUTION
            + map_height
        )


        # ----------------------------------------------------
        # Map crop → video coordinates
        # ----------------------------------------------------

        px = (
            (px - x_min)
            * scale
            + offset_x
        ).astype(np.int32)

        py = (
            (py - y_min)
            * scale
            + offset_y
        ).astype(np.int32)


        # ----------------------------------------------------
        # Valid points
        # ----------------------------------------------------

        valid = (
            (px >= 0) &
            (px < VIDEO_W) &
            (py >= 0) &
            (py < VIDEO_H)
        )

        px = px[valid]
        py = py[valid]


        # ----------------------------------------------------
        # Draw LiDAR
        # ----------------------------------------------------

        frame[py, px] = (
            255, 100, 0
        )


        # ====================================================
        # Draw 3D bounding boxes
        # ====================================================

        sample_token = sd["sample_token"]

        frame_annotations = sample_annotations.get(
            sample_token,
            []
        )

        for ann in frame_annotations:

            # -----------------------------------------------
            # Box corners in GLOBAL coordinates
            # -----------------------------------------------

            corners_global = get_box_corners(
                ann
            )

            # -----------------------------------------------
            # GLOBAL → MAP PIXELS
            # -----------------------------------------------

            box_px = (
                corners_global[:, 0]
                / RESOLUTION
            )

            box_py = (
                -corners_global[:, 1]
                / RESOLUTION
                + map_height
            )

            # -----------------------------------------------
            # MAP → VIDEO coordinates
            # -----------------------------------------------

            box_x = (
                (box_px - x_min)
                * scale
                + offset_x
            ).astype(np.int32)

            box_y = (
                (box_py - y_min)
                * scale
                + offset_y
            ).astype(np.int32)

            box_pts = np.column_stack(
                (box_x, box_y)
            )


            # -----------------------------------------------
            # Bottom footprint
            # -----------------------------------------------

            bottom = box_pts[4:8]

            cv2.polylines(
                frame,
                [bottom],
                True,
                (0, 0, 255),
                2,
                cv2.LINE_AA
            )


        # ----------------------------------------------------
        # Draw trajectory
        # ----------------------------------------------------

        if frame_idx > 0:

            trajectory = np.column_stack(
                (
                    traj_x[:frame_idx + 1],
                    traj_y[:frame_idx + 1]
                )
            )

            cv2.polylines(
                frame,
                [trajectory],
                False,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )


        # ----------------------------------------------------
        # Ego vehicle
        # ----------------------------------------------------

        ex = traj_x[frame_idx]
        ey = traj_y[frame_idx]

        cv2.circle(
            frame,
            (ex, ey),
            8,
            (0, 0, 255),
            -1
        )


        # ----------------------------------------------------
        # Scene information
        # ----------------------------------------------------

        cv2.rectangle(
            frame,
            (10, 10),
            (550, 95),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            frame,
            f"Scene {scene_idx + 1}/{len(scenes)} | "
            f"{location}",
            (25, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Frame {frame_idx + 1}/{len(records)} | "
            f"Objects: {len(frame_annotations)}",
            (25, 72),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )


        # ----------------------------------------------------
        # Write
        # ----------------------------------------------------

        writer.write(frame)

        total_frames += 1


    # --------------------------------------------------------
    # Release map memory before next scene
    # --------------------------------------------------------

    map_img.close()

    print(
        f"Completed scene "
        f"{scene_idx + 1}/{len(scenes)}"
    )


# ============================================================
# 6. Finish
# ============================================================

writer.release()

print("\n======================================")
print("ALL SCENES COMPLETE")
print("======================================")

print(
    "Total frames:",
    total_frames
)

print(
    "FPS:",
    FPS
)

print(
    "Duration:",
    round(total_frames / FPS, 2),
    "seconds"
)

print("Output:")
print(output_video)

In [ ]:
# ============================================================
# Load nuScenes 3D object annotations
# ============================================================

with open(
    os.path.join(meta_path, "sample_annotation.json")
) as f:
    annotations = json.load(f)

print("Annotations:", len(annotations))

# Group annotations by sample
sample_annotations = {}

for ann in annotations:
    sample_annotations.setdefault(
        ann["sample_token"], []
    ).append(ann)

print(
    "Samples containing annotations:",
    len(sample_annotations)
)

# Example: first LiDAR sample
first_sd = scene_lidar_records[0]

first_sample_token = first_sd["sample_token"]

print(
    "Annotations in first sample:",
    len(
        sample_annotations.get(
            first_sample_token,
            []
        )
    )
)

In [ ]:
# ============================================================
# Test 3D bounding boxes for one LiDAR sample
# ============================================================

def get_box_corners(annotation):
    """
    Return 8 corners of a nuScenes 3D box in GLOBAL coordinates.
    """

    center = np.asarray(
        annotation["translation"],
        dtype=np.float64
    )

    # nuScenes size = [width, length, height]
    w, l, h = annotation["size"]

    # Local box corners
    corners = np.array([
        [ l/2,  w/2,  h/2],
        [ l/2, -w/2,  h/2],
        [-l/2, -w/2,  h/2],
        [-l/2,  w/2,  h/2],

        [ l/2,  w/2, -h/2],
        [ l/2, -w/2, -h/2],
        [-l/2, -w/2, -h/2],
        [-l/2,  w/2, -h/2],
    ])

    # Box rotation
    R = quat_to_rotmat(
        annotation["rotation"]
    )

    # Local → global
    corners_global = (
        corners @ R.T
        + center
    )

    return corners_global


# ------------------------------------------------------------
# First sample annotations
# ------------------------------------------------------------

first_sample_token = (
    scene_lidar_records[0]["sample_token"]
)

anns = sample_annotations[
    first_sample_token
]

print("Objects:", len(anns))


# ------------------------------------------------------------
# Convert corners to map pixels
# ------------------------------------------------------------

all_box_pixels = []

for ann in anns:

    corners_global = get_box_corners(
        ann
    )

    px = (
        corners_global[:, 0]
        / RESOLUTION
    )

    py = (
        -corners_global[:, 1]
        / RESOLUTION
        + map_height
    )

    all_box_pixels.append(
        np.column_stack((px, py))
    )


print(
    "Boxes converted:",
    len(all_box_pixels)
)

print(
    "First box corners:\n",
    all_box_pixels[0]
)

In [ ]:
# ============================================================
# Create ego pose lookup
# ============================================================

ego_pose_by_token = {
    pose["token"]: pose
    for pose in ego_poses
}

print("Ego pose lookup:", len(ego_pose_by_token))

In [ ]:
# ============================================================
# Draw annotated 3D bounding boxes on the first sample map
# ============================================================

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Load map
map_img = np.array(
    Image.open(
        os.path.join(data_path, map_filename)
    )
)

# Convert grayscale → BGR for OpenCV
if len(map_img.shape) == 2:
    frame = cv2.cvtColor(map_img, cv2.COLOR_GRAY2BGR)
else:
    frame = map_img.copy()

# ------------------------------------------------------------
# Draw each annotation
# ------------------------------------------------------------

for ann in anns:

    corners = get_box_corners(ann)

    # Global → map pixels
    px = corners[:, 0] / RESOLUTION
    py = -corners[:, 1] / RESOLUTION + map_height

    pts = np.column_stack((px, py)).astype(np.int32)

    # Bottom face
    bottom = pts[4:8]

    # Top face
    top = pts[0:4]

    # Draw bottom rectangle
    cv2.polylines(
        frame,
        [bottom],
        isClosed=True,
        color=(0, 0, 255),
        thickness=3
    )

    # Draw top rectangle
    cv2.polylines(
        frame,
        [top],
        isClosed=True,
        color=(0, 255, 0),
        thickness=2
    )

    # Vertical edges
    for i in range(4):
        cv2.line(
            frame,
            tuple(top[i]),
            tuple(bottom[i]),
            (255, 0, 0),
            2
        )

# ------------------------------------------------------------
# Crop around the first LiDAR frame
# ------------------------------------------------------------

sd = scene_lidar_records[0]

ego_pose = ego_pose_by_token[
    sd["ego_pose_token"]
]

ego_x = ego_pose["translation"][0]
ego_y = ego_pose["translation"][1]

# Crop size
PAD_METERS = 100

x_min = ego_x - PAD_METERS
x_max = ego_x + PAD_METERS
y_min = ego_y - PAD_METERS
y_max = ego_y + PAD_METERS

px_min = int(x_min / RESOLUTION)
px_max = int(x_max / RESOLUTION)

py_max = int(-y_min / RESOLUTION + map_height)
py_min = int(-y_max / RESOLUTION + map_height)

crop = frame[
    py_min:py_max,
    px_min:px_max
]

# Resize for display
display = cv2.resize(
    crop,
    (1000, 800)
)

plt.figure(figsize=(12, 9))
plt.imshow(
    cv2.cvtColor(display, cv2.COLOR_BGR2RGB)
)
plt.axis("off")
plt.title("nuScenes — Annotated 3D Bounding Boxes")
plt.show()

In [41]:
# ============================================================
# ALL 404 nuScenes LiDAR SAMPLES
# Scene-aware map visualization + 3D bounding boxes (v3 - self-contained)
# ============================================================
#
# FULLY SELF-CONTAINED: this cell loads everything it needs from disk
# and defines every helper function itself. It does NOT depend on any
# earlier cell in the notebook (other than data_path being reachable),
# so it can be run standalone, even right after a kernel restart.
#
# Features:
#   1. LiDAR points colored by height (TURBO colormap)
#   2. Boxes colored by category + text label + legend
#   3. Ego vehicle drawn as a heading arrow (not a plain dot)
#   4. Trajectory trail fades from dim -> bright with recency
#   5. Points drawn as small anti-aliased dots (not 1px)
#   6. Semi-transparent info panel (alpha blend) instead of solid black
# ============================================================

import os
import json
import cv2
import numpy as np
from PIL import Image

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

data_path = "/kaggle/input/datasets/aadimator/nuscenes-mini"
meta_path = os.path.join(data_path, "v1.0-mini")

FPS = 10
VIDEO_W = 1200
VIDEO_H = 900
RESOLUTION = 0.1
PAD_METERS = 30

TRAIL_LENGTH = 40          # how many past frames to show in the fading trail
Z_MIN, Z_MAX = -2.0, 4.0   # height range (meters, relative to LiDAR) for color mapping

output_video = "/kaggle/working/nuscenes_all_scenes_lidar_map_boxes_v3.mp4"


# ============================================================
# 1. Load ALL required nuScenes metadata
# ============================================================

def load_json(name):
    with open(os.path.join(meta_path, name)) as f:
        return json.load(f)

samples = load_json("sample.json")
sample_data = load_json("sample_data.json")
ego_poses = load_json("ego_pose.json")
calibrated_sensors = load_json("calibrated_sensor.json")
scenes = load_json("scene.json")
logs = load_json("log.json")
maps = load_json("map.json")
instances = load_json("instance.json")
categories = load_json("category.json")
sample_annotation_records = load_json("sample_annotation.json")

# Annotations grouped by sample_token
sample_annotations = {}
for ann in sample_annotation_records:
    sample_annotations.setdefault(ann["sample_token"], []).append(ann)


# ------------------------------------------------------------
# Fast lookups
# ------------------------------------------------------------

scene_lookup = {s["token"]: s for s in scenes}
log_lookup = {l["token"]: l for l in logs}
sample_lookup = {s["token"]: s for s in samples}

pose_lookup = {p["token"]: p for p in ego_poses}
calib_lookup = {c["token"]: c for c in calibrated_sensors}

instance_lookup = {i["token"]: i for i in instances}
category_lookup = {c["token"]: c["name"] for c in categories}


# ------------------------------------------------------------
# Core geometry helpers (quaternion rotation, coordinate transforms)
# ------------------------------------------------------------

def quat_to_rotmat(q):
    """
    nuScenes quaternion format: [w, x, y, z]
    Returns 3x3 rotation matrix.
    """
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)]
    ])


def transform_points(points, translation, rotation):
    """
    p_out = R @ p + t   (applied to an (N,3) array of points)
    """
    R = quat_to_rotmat(rotation)
    return points @ R.T + np.asarray(translation)


def get_box_corners(annotation):
    """
    Return the 8 corners of a nuScenes 3D box in GLOBAL coordinates.

    Corner order (matters for downstream drawing code):
      0: top-front-left     4: bottom-front-left
      1: top-front-right    5: bottom-front-right
      2: top-back-right     6: bottom-back-right
      3: top-back-left      7: bottom-back-left

    So corners[4:8] traces a proper bottom-face polygon, and the
    midpoint of corners[4] & corners[5] is the front-center of the
    footprint (used for the heading arrow).
    """
    center = np.asarray(annotation["translation"], dtype=np.float64)

    # nuScenes size = [width, length, height]
    w, l, h = annotation["size"]

    corners_local = np.array([
        [ l/2,  w/2,  h/2],
        [ l/2, -w/2,  h/2],
        [-l/2, -w/2,  h/2],
        [-l/2,  w/2,  h/2],

        [ l/2,  w/2, -h/2],
        [ l/2, -w/2, -h/2],
        [-l/2, -w/2, -h/2],
        [-l/2,  w/2, -h/2],
    ])

    R = quat_to_rotmat(annotation["rotation"])
    corners_global = corners_local @ R.T + center

    return corners_global


def get_category_name(ann):
    """
    Resolve a human-readable category name for an annotation dict.
    Falls back gracefully if the structure differs from raw nuScenes JSON.
    """
    if "category_name" in ann:
        return ann["category_name"]

    inst = instance_lookup.get(ann.get("instance_token"))
    if inst is None:
        return "unknown"

    return category_lookup.get(inst.get("category_token"), "unknown")


# ------------------------------------------------------------
# Category -> color map (BGR)
# ------------------------------------------------------------

CATEGORY_COLOR_RULES = [
    ("vehicle.car",              (0, 255, 255)),   # yellow
    ("vehicle.truck",            (0, 140, 255)),   # orange
    ("vehicle.bus",              (0, 90, 255)),     # deep orange
    ("vehicle.construction",     (0, 60, 180)),
    ("vehicle.trailer",          (0, 200, 200)),
    ("vehicle.motorcycle",       (255, 0, 200)),
    ("vehicle.bicycle",          (0, 255, 0)),      # green
    ("human.pedestrian",         (255, 0, 255)),    # magenta
    ("movable_object.barrier",   (180, 180, 180)),
    ("movable_object.trafficcone", (0, 165, 255)),
    ("static_object",            (120, 120, 120)),
]

DEFAULT_BOX_COLOR = (0, 0, 255)  # red fallback


def get_category_color(category_name):
    for prefix, color in CATEGORY_COLOR_RULES:
        if category_name.startswith(prefix):
            return color
    return DEFAULT_BOX_COLOR


def short_label(category_name):
    # "vehicle.car" -> "car", "human.pedestrian.adult" -> "adult"
    return category_name.split(".")[-1]


# ============================================================
# 2. Group LIDAR keyframes by scene
# ============================================================

scene_lidar = {s["token"]: [] for s in scenes}

for sd in sample_data:

    if not sd["is_key_frame"]:
        continue

    if not sd["filename"].startswith("samples/LIDAR_TOP"):
        continue

    sample = sample_lookup.get(sd["sample_token"])

    if sample is None:
        continue

    scene_token = sample["scene_token"]
    scene_lidar[scene_token].append(sd)

for scene_token in scene_lidar:
    scene_lidar[scene_token].sort(key=lambda x: x["timestamp"])

print("Scenes:", len(scenes))
print("Total LiDAR keyframes:", sum(len(v) for v in scene_lidar.values()))


# ============================================================
# 3. Map lookup
# ============================================================

log_to_map = {}
for m in maps:
    for log_token in m["log_tokens"]:
        log_to_map[log_token] = m["filename"]


# ============================================================
# 4. Helpers: quaternion -> yaw, alpha-blended panel, colormap
# ============================================================

def quaternion_yaw(q):
    """
    nuScenes quaternion is [w, x, y, z].
    Returns yaw (rotation about Z) in radians, in the GLOBAL frame,
    matching the convention used by (px, py) = (x/res, -y/res + H).
    """
    w, x, y, z = q
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    return np.arctan2(siny_cosp, cosy_cosp)


def draw_translucent_panel(frame, x0, y0, x1, y1, color=(0, 0, 0), alpha=0.55):
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)


def height_to_color(z_values):
    """
    Map an array of z (height) values to BGR colors using TURBO colormap.
    """
    h_norm = np.clip((z_values - Z_MIN) / (Z_MAX - Z_MIN), 0, 1)
    h_uint8 = (h_norm * 255).astype(np.uint8).reshape(-1, 1)
    colored = cv2.applyColorMap(h_uint8, cv2.COLORMAP_TURBO)
    return colored.reshape(-1, 3)


def draw_points(frame, xs, ys, colors, radius=1):
    """
    Draw points as small filled circles (anti-aliased-ish via radius>=1)
    instead of raw 1px assignment, so they survive video compression.
    """
    h, w = frame.shape[:2]
    for x, y, c in zip(xs, ys, colors):
        cv2.circle(frame, (int(x), int(y)), radius, tuple(int(v) for v in c), -1, cv2.LINE_AA)


def draw_legend(frame, used_categories, x0=10, y0=110):
    if not used_categories:
        return

    row_h = 22
    panel_h = row_h * len(used_categories) + 16
    draw_translucent_panel(frame, x0, y0, x0 + 190, y0 + panel_h, (0, 0, 0), 0.55)

    for i, cat in enumerate(sorted(used_categories)):
        color = get_category_color(cat)
        cy = y0 + 18 + i * row_h
        cv2.rectangle(frame, (x0 + 10, cy - 8), (x0 + 26, cy + 4), color, -1)
        cv2.putText(
            frame, short_label(cat), (x0 + 34, cy + 3),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA
        )


# ============================================================
# 5. Prepare video writer
# ============================================================

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (VIDEO_W, VIDEO_H)
)

if not writer.isOpened():
    raise RuntimeError("Could not open video writer")


# ============================================================
# 6. Process scenes
# ============================================================

total_frames = 0

for scene_idx, scene in enumerate(scenes):

    scene_token = scene["token"]
    records = scene_lidar[scene_token]

    if not records:
        print(f"Skipping scene {scene_idx + 1}: no LiDAR frames")
        continue

    # --------------------------------------------------------
    # Scene / log information
    # --------------------------------------------------------

    log = log_lookup[scene["log_token"]]
    location = log["location"]
    map_filename = log_to_map.get(scene["log_token"])

    print(f"\nScene {scene_idx + 1}/{len(scenes)}")
    print("Location:", location)
    print("Frames:", len(records))
    print("Map:", map_filename)

    if map_filename is None:
        print("No map found — skipping.")
        continue

    # --------------------------------------------------------
    # Load relevant map
    # --------------------------------------------------------

    map_path = os.path.join(data_path, map_filename)
    Image.MAX_IMAGE_PIXELS = None
    map_img = Image.open(map_path)
    map_width, map_height = map_img.size

    # --------------------------------------------------------
    # Scene ego trajectory
    # --------------------------------------------------------

    scene_trajectory_global = np.array([
        pose_lookup[sd["ego_pose_token"]]["translation"][:2]
        for sd in records
    ])

    scene_yaws = np.array([
        quaternion_yaw(pose_lookup[sd["ego_pose_token"]]["rotation"])
        for sd in records
    ])

    trajectory_px = scene_trajectory_global[:, 0] / RESOLUTION
    trajectory_py = -scene_trajectory_global[:, 1] / RESOLUTION + map_height

    # --------------------------------------------------------
    # Crop map around THIS scene
    # --------------------------------------------------------

    pad_px = int(PAD_METERS / RESOLUTION)

    x_min = max(0, int(trajectory_px.min()) - pad_px)
    x_max = min(map_width, int(trajectory_px.max()) + pad_px)
    y_min = max(0, int(trajectory_py.min()) - pad_px)
    y_max = min(map_height, int(trajectory_py.max()) + pad_px)

    map_crop = map_img.crop((x_min, y_min, x_max, y_max))

    # --------------------------------------------------------
    # Resize map to fit fixed video canvas
    # --------------------------------------------------------

    crop_w, crop_h = map_crop.size
    scale = min(VIDEO_W / crop_w, VIDEO_H / crop_h)
    resized_w = int(crop_w * scale)
    resized_h = int(crop_h * scale)

    map_np = np.array(map_crop)
    map_resized = cv2.resize(map_np, (resized_w, resized_h), interpolation=cv2.INTER_AREA)

    base_frame = np.zeros((VIDEO_H, VIDEO_W, 3), dtype=np.uint8)
    map_bgr = cv2.cvtColor(map_resized, cv2.COLOR_GRAY2BGR)

    offset_x = (VIDEO_W - resized_w) // 2
    offset_y = (VIDEO_H - resized_h) // 2

    base_frame[
        offset_y:offset_y + resized_h,
        offset_x:offset_x + resized_w
    ] = map_bgr

    # --------------------------------------------------------
    # Convert trajectory to video coordinates
    # --------------------------------------------------------

    traj_x = ((trajectory_px - x_min) * scale + offset_x).astype(np.int32)
    traj_y = ((trajectory_py - y_min) * scale + offset_y).astype(np.int32)

    # ========================================================
    # Render scene frames
    # ========================================================

    for frame_idx, sd in enumerate(records):

        frame = base_frame.copy()

        # ----------------------------------------------------
        # Load LiDAR
        # ----------------------------------------------------

        lidar_path = os.path.join(data_path, sd["filename"])
        points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
        xyz = points[:, :3]
        z_raw = xyz[:, 2].copy()  # height in LiDAR/ego frame, used for coloring

        # ----------------------------------------------------
        # LiDAR -> Ego
        # ----------------------------------------------------

        calib = calib_lookup[sd["calibrated_sensor_token"]]
        ego_points = transform_points(xyz, calib["translation"], calib["rotation"])

        # ----------------------------------------------------
        # Ego -> Global
        # ----------------------------------------------------

        pose = pose_lookup[sd["ego_pose_token"]]
        global_points = transform_points(ego_points, pose["translation"], pose["rotation"])

        # ----------------------------------------------------
        # Global -> Map pixels
        # ----------------------------------------------------

        px = global_points[:, 0] / RESOLUTION
        py = -global_points[:, 1] / RESOLUTION + map_height

        # ----------------------------------------------------
        # Map crop -> video coordinates
        # ----------------------------------------------------

        px = ((px - x_min) * scale + offset_x).astype(np.int32)
        py = ((py - y_min) * scale + offset_y).astype(np.int32)

        # ----------------------------------------------------
        # Valid points
        # ----------------------------------------------------

        valid = (px >= 0) & (px < VIDEO_W) & (py >= 0) & (py < VIDEO_H)
        px = px[valid]
        py = py[valid]
        z_valid = z_raw[valid]

        # ----------------------------------------------------
        # Draw LiDAR (colored by height, small dots)
        # ----------------------------------------------------

        if len(px) > 0:
            point_colors = height_to_color(z_valid)
            draw_points(frame, px, py, point_colors, radius=1)

        # ====================================================
        # Draw 3D bounding boxes (colored by category + labels)
        # ====================================================

        sample_token = sd["sample_token"]
        frame_annotations = sample_annotations.get(sample_token, [])
        used_categories = set()

        for ann in frame_annotations:

            category_name = get_category_name(ann)
            used_categories.add(category_name)
            box_color = get_category_color(category_name)

            # -------------------------------------------------
            # Box corners in GLOBAL coordinates
            # -------------------------------------------------

            corners_global = get_box_corners(ann)

            box_px = corners_global[:, 0] / RESOLUTION
            box_py = -corners_global[:, 1] / RESOLUTION + map_height

            box_x = ((box_px - x_min) * scale + offset_x).astype(np.int32)
            box_y = ((box_py - y_min) * scale + offset_y).astype(np.int32)

            box_pts = np.column_stack((box_x, box_y))

            # -------------------------------------------------
            # Bottom footprint
            # -------------------------------------------------

            bottom = box_pts[4:8]

            cv2.polylines(frame, [bottom], True, box_color, 2, cv2.LINE_AA)

            # -------------------------------------------------
            # Heading tick: from footprint center toward the
            # front edge (corners 4-5 assumed "front" pair;
            # adjust indices if your get_box_corners orders
            # corners differently)
            # -------------------------------------------------

            center = bottom.mean(axis=0).astype(np.int32)
            front_mid = ((bottom[0] + bottom[1]) / 2).astype(np.int32)
            cv2.arrowedLine(
                frame, tuple(center), tuple(front_mid),
                box_color, 2, cv2.LINE_AA, tipLength=0.4
            )

            # -------------------------------------------------
            # Label
            # -------------------------------------------------

            label_pos = (int(bottom[:, 0].min()), int(bottom[:, 1].min()) - 6)
            cv2.putText(
                frame, short_label(category_name), label_pos,
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, box_color, 1, cv2.LINE_AA
            )

        # ----------------------------------------------------
        # Draw fading trajectory trail (dim -> bright)
        # ----------------------------------------------------

        if frame_idx > 0:
            start = max(0, frame_idx - TRAIL_LENGTH)
            trail_x = traj_x[start:frame_idx + 1]
            trail_y = traj_y[start:frame_idx + 1]
            n = len(trail_x)

            for i in range(1, n):
                t = i / max(1, n - 1)  # 0 (old) -> 1 (recent)
                brightness = int(60 + t * 195)
                color = (0, brightness, 0)
                pt1 = (int(trail_x[i - 1]), int(trail_y[i - 1]))
                pt2 = (int(trail_x[i]), int(trail_y[i]))
                cv2.line(frame, pt1, pt2, color, 2, cv2.LINE_AA)

        # ----------------------------------------------------
        # Ego vehicle as a heading arrow (not a plain dot)
        # ----------------------------------------------------

        ex, ey = int(traj_x[frame_idx]), int(traj_y[frame_idx])
        yaw = scene_yaws[frame_idx]

        arrow_len = 18
        tip = (int(ex + arrow_len * np.cos(yaw)), int(ey - arrow_len * np.sin(yaw)))

        cv2.circle(frame, (ex, ey), 6, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.arrowedLine(frame, (ex, ey), tip, (0, 0, 255), 3, cv2.LINE_AA, tipLength=0.5)

        # ----------------------------------------------------
        # Scene information (semi-transparent panel)
        # ----------------------------------------------------

        draw_translucent_panel(frame, 10, 10, 550, 95, (0, 0, 0), 0.55)

        cv2.putText(
            frame,
            f"Scene {scene_idx + 1}/{len(scenes)} | {location}",
            (25, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Frame {frame_idx + 1}/{len(records)} | Objects: {len(frame_annotations)}",
            (25, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA
        )

        # ----------------------------------------------------
        # Category legend (bottom-left growing panel)
        # ----------------------------------------------------

        draw_legend(frame, used_categories, x0=10, y0=110)

        # ----------------------------------------------------
        # Write
        # ----------------------------------------------------

        writer.write(frame)
        total_frames += 1

    # --------------------------------------------------------
    # Release map memory before next scene
    # --------------------------------------------------------

    map_img.close()
    print(f"Completed scene {scene_idx + 1}/{len(scenes)}")


# ============================================================
# 7. Finish
# ============================================================

writer.release()

print("\n======================================")
print("ALL SCENES COMPLETE")
print("======================================")
print("Total frames:", total_frames)
print("FPS:", FPS)
print("Duration:", round(total_frames / FPS, 2), "seconds")
print("Output:")
print(output_video)

Scenes: 10
Total LiDAR keyframes: 404

Scene 1/10
Location: singapore-onenorth
Frames: 39
Map: maps/53992ee3023e5494b90c316c183be829.png
Completed scene 1/10

Scene 2/10
Location: boston-seaport
Frames: 40
Map: maps/36092f0b03a857c6a3403e25b4b7aab3.png
Completed scene 2/10

Scene 3/10
Location: boston-seaport
Frames: 41
Map: maps/36092f0b03a857c6a3403e25b4b7aab3.png
Completed scene 3/10

Scene 4/10
Location: boston-seaport
Frames: 41
Map: maps/36092f0b03a857c6a3403e25b4b7aab3.png
Completed scene 4/10

Scene 5/10
Location: boston-seaport
Frames: 41
Map: maps/36092f0b03a857c6a3403e25b4b7aab3.png
Completed scene 5/10

Scene 6/10
Location: singapore-queenstown
Frames: 40
Map: maps/93406b464a165eaba6d9de76ca09f5da.png
Completed scene 6/10

Scene 7/10
Location: singapore-queenstown
Frames: 41
Map: maps/93406b464a165eaba6d9de76ca09f5da.png
Completed scene 7/10

Scene 8/10
Location: singapore-hollandvillage
Frames: 41
Map: maps/37819e65e09e5547b8a3ceaefba56bb2.png
Completed scene 8/10

Scene 9